# 02 — Render ShapeNetCore

Colab Free quick test: chuẩn bị mesh synthetic và render 1 mesh/1 view. Notebook kiểm tra GPU, mount Drive khi ở Colab và chạy render hai lần để xác minh resume.

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
STARTED = time.perf_counter()
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents] if (p / 'configs' / 'quick_test.yaml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Không tìm thấy repository; hãy mở notebook từ repo hoặc thư mục cha.')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Cài dependency thất bại; Restart runtime rồi Run all lại.') from exc
import torch
CUDA = torch.cuda.is_available()
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount(os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive'))
    DRIVE_ROOT = Path(os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive')) / 'MyDrive' / 'image-to-3d-shapenet'
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except ImportError:
    pass
HF_TOKEN_AVAILABLE = bool(os.getenv('HF_TOKEN'))
try:
    from google.colab import userdata
    HF_TOKEN_AVAILABLE = HF_TOKEN_AVAILABLE or bool(userdata.get('HF_TOKEN'))
except Exception:
    pass
print({'project': str(PROJECT_ROOT), 'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'hf_token_available': HF_TOKEN_AVAILABLE, 'drive': str(DRIVE_ROOT) if DRIVE_ROOT else None})

In [ ]:
from dataclasses import replace
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from image_to_3d_shapenet import load_config
from image_to_3d_shapenet.dataset import prepare_dataset

QUICK_TEST = True
MAX_MESHES = 1  # Chọn 1–5
config_name = 'quick_test.yaml' if QUICK_TEST else 'chair_colab.yaml'
config = load_config(PROJECT_ROOT / 'configs' / config_name)
config = replace(config, dataset=replace(config.dataset, views_per_mesh=1), render=replace(config.render, max_render_meshes=MAX_MESHES))
if QUICK_TEST:
    preparation = prepare_dataset(config, PROJECT_ROOT, synthetic=True)
    print(preparation)
else:
    print('Dùng mesh ShapeNetCore đã chuẩn bị bằng scripts/prepare_dataset.py')

In [ ]:
from image_to_3d_shapenet.rendering import render_dataset

try:
    render_report = render_dataset(config, PROJECT_ROOT)
    resumed_report = render_dataset(config, PROJECT_ROOT)  # manifest skip/resume
except Exception as exc:
    raise RuntimeError('Render thất bại; kiểm tra EGL/pyrender. Backend software phải tự fallback.') from exc
render_report

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

render_root = Path(render_report['render_root'])
manifest = json.loads(Path(render_report['manifest']).read_text(encoding='utf-8'))
samples = list(manifest['samples'].values())[:5]
for sample in samples:
    sample_dir = render_root / sample['sample_directory']
    rgb = np.asarray(Image.open(sample_dir / 'rgb.png'))
    mask = np.asarray(Image.open(sample_dir / 'mask.png'))
    depth = np.load(sample_dir / 'depth.npy', allow_pickle=False)
    normal = np.asarray(Image.open(sample_dir / 'normal.png'))
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for axis, image, title in zip(axes, [rgb, mask, depth, normal], ['RGB', 'Mask', 'Depth', 'Normal']):
        axis.imshow(image, cmap='gray' if image.ndim == 2 else None)
        axis.set_title(title)
        axis.axis('off')
    fig.suptitle(sample['sample_id'] + ' — ' + sample['backend'])
    plt.show()
disk = shutil.disk_usage(PROJECT_ROOT)
peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if CUDA else 0.0
print({'elapsed_seconds': round(time.perf_counter() - STARTED, 2), 'peak_vram_gb': round(peak_vram, 3), 'disk_free_gb': round(disk.free / 1024**3, 2), 'resume_skipped': resumed_report['skipped']})
if DRIVE_ROOT:
    target = DRIVE_ROOT / 'renders' / 'manifest.json'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path(render_report['manifest']), target)